# MENTORA — Train M1: Knowledge Gap Classifier (DeBERTa-v3-base)

Target: **F1 (micro) >= 0.85**.

**Note:** our starter `question_bank.csv` has 70 questions total (target is 150-300 per subject — see datasets/SOURCES.md). Expect this run to be a pipeline smoke test more than a metric-hitting run until the bank is scaled up with real past-paper questions.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

### Optional — Weights & Biases logging
Nice for live loss curves / a thesis screenshot. Skip entirely if you'd rather keep things simple — nothing below strictly needs it.

In [ ]:
# !pip install -q wandb
# import wandb
# wandb.login()  # paste your free-tier API key when prompted, once per session

## 1. Load and tokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd, numpy as np

df = pd.read_csv(f'{DATA}/processed/m1/question_bank.csv')
topics = sorted(df['topic'].unique())
NUM_TOPICS = len(topics)
topic_to_idx = {t: i for i, t in enumerate(topics)}

def to_multihot(topic_str):
    vec = [0.0] * NUM_TOPICS
    for t in str(topic_str).split('|'):  # our data is single-topic; this also handles multi-topic if added later
        vec[topic_to_idx[t]] = 1.0
    return vec

df['labels'] = df['topic'].apply(to_multihot)

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
def tokenize(batch):
    enc = tokenizer(batch['question_text'], truncation=True, padding='max_length', max_length=128)
    enc['labels'] = batch['labels']
    return enc

ds = Dataset.from_pandas(df[['question_text', 'labels']])
ds = ds.train_test_split(test_size=0.15, seed=42)
ds = ds.map(tokenize, batched=True)

## 2. Model + metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base', num_labels=NUM_TOPICS, problem_type='multi_label_classification'
)

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {'f1_micro': f1_score(labels, preds, average='micro', zero_division=0)}

## 3. Train — checkpoints every epoch to Drive, resumable

In [ ]:
import glob

OUT_DIR = f'{MODELS}/model1_gap_classifier'

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    fp16=True,
    report_to='wandb' if 'wandb' in dir() else 'none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

existing_checkpoints = glob.glob(f'{OUT_DIR}/checkpoint-*')
resume = sorted(existing_checkpoints)[-1] if existing_checkpoints else None
trainer.train(resume_from_checkpoint=resume)

## 4. Save final model

In [ ]:
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')
print(trainer.evaluate())

## Target metric: F1 (micro) >= 0.85

If it plateaus lower:
- Sweep the classification threshold (0.3/0.4/0.5/0.6) on validation
- Check per-topic example counts — merge near-unlearnable rare subtopics into their parent
- Highest-leverage fix: grow the M1 question bank past the 70-question starter set (also directly benefits M2)